# Chef Agent

Use this notebook to prototype the main Personal Chef Agent workflow.

In [1]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent

sys.path.insert(0, str(PROJECT_ROOT))

print(sys.path[0])

c:\Users\praya\Agentic_PersonalChef


In [2]:
import langchain
import groq
import dotenv
import sounddevice
import scipy

print("All requirements installed successfully")

All requirements installed successfully


In [3]:
from langchain_groq import ChatGroq
from dotenv import load_dotenv

print("Environment working")

Environment working


In [6]:

from config import llm
from langchain.tools import tool
from typing import Dict, Any
from tavily import TavilyClient

tavily_client=TavilyClient()

@tool
def web_search(query:str)->Dict[str,Any]:
    """Search the web for information"""

    return tavily_client.search(query)

In [7]:
system_prompt="""You are a personal agentic chef. Your task is to create a personalised recipe 
with the leftover food items provided by the user. Use the web search to provide the recipe to the user.
If the user asks for recipe instruction provide them with the same."""

In [8]:
from langchain.messages import HumanMessage
from langchain.agents import create_agent
from langgraph.checkpoint.memory import InMemorySaver

agent=create_agent(
    model=llm,
    tools=[web_search],
    system_prompt=system_prompt,
    checkpointer=InMemorySaver()
)

In [13]:
from langchain.messages import HumanMessage

config={"configurbale":{"thread_id":"1"}}

config = {"configurable": {"thread_id": "1"}}

response = agent.invoke({"messages": [HumanMessage(content="I have some leftover milk and bread left. What can i make?")]},
                        config)

print(response['messages'][-1].content)


You can make a variety of dishes using leftover milk and bread. Here are a few ideas:

1. Milk Toast: A simple dessert made by toasting bread, mixing milk with sugar and vanilla, and then soaking the toast in the milk mixture.
2. Breakfast Casserole: A casserole made by layering toasted or stale bread, eggs, milk, and spices/herbs of choice.
3. Bread Pudding: A creamy pudding made by soaking leftover bread in a mixture of milk, sugar, and eggs, and then baking until set.

Here's a simple recipe for Milk Toast:

Ingredients:

* 2 pieces of bread
* 1/2 cup of milk
* 1 tsp of sugar
* 1/2 tsp of vanilla extract
* Butter for toasting

Instructions:

1. Cut the edges of the bread and spread butter on each piece.
2. Toast the bread until golden brown.
3. Mix milk, sugar, and vanilla extract in a bowl.
4. Soak the toasted bread in the milk mixture until the bread absorbs the liquid.
5. Serve warm and enjoy!

You can find more recipes and detailed instructions by searching online or checking ou

In [33]:
from langgraph.checkpoint.memory import InMemorySaver

from langchain.agents import create_agent
from langchain_core.messages import HumanMessage

from config import vision_llm

image_agent = create_agent(
    model=vision_llm,
    tools=[web_search],
    checkpointer=InMemorySaver()
)

config = {
    "configurable": {
        "thread_id": "inventory-session-1"
    }
}

In [28]:
from ipywidgets import FileUpload
from IPython.display import display

uploader = FileUpload(
    accept="image/*",
    multiple=False
)

display(uploader)

FileUpload(value=(), accept='image/*', description='Upload')

In [30]:
import base64

if not uploader.value:
    raise ValueError("Please upload an image first.")

uploaded_file = uploader.value[0]

image_bytes = uploaded_file["content"]

img_b64 = base64.b64encode(image_bytes).decode("utf-8")

mime_type = uploaded_file["type"]

image_url = f"data:{mime_type};base64,{img_b64}"

In [37]:
from langchain_core.messages import HumanMessage

message = HumanMessage(
    content=[
        {
            "type": "text",
            "text": """
            Analyze this fridge inventory.

            Identify all visible ingredients and food items.
            """
        },
        {
            "type": "image_url",
            "image_url": {
                "url": image_url
            }
        }
    ]
)

In [38]:
from config import vision_llm

vision_response = vision_llm.invoke([message])

inventory_text = vision_response.content

print(inventory_text)

The visible ingredients and food items in the fridge are:

* Breads:
	+ Bread (wrapped in plastic)
	+ Bagels
* Dairy:
	+ Cottage cheese
	+ Cheese (block)
	+ Eggs
	+ Milk ( Fairlife brand)
* Fruits:
	+ Apples (2)
	+ Lemons (3-4)
* Proteins:
	+ Turkey chili
	+ Tuna (in cans)
* Prepared foods:
	+ Leftovers (in glass containers)
* Snacks:
	+ Granola or cereal (Chobani Zero Sugar)
* Beverages:
	+ Juice or milk alternative (purple carton)

Note that some items may be partially obscured or difficult to identify, but these are the most clearly visible ingredients and food items.


In [43]:
response = agent.invoke(
    {
        "messages": [
            HumanMessage(
                content=f"""
                Here are ingredients available in my fridge:

                {inventory_text}

                Based on these ingredients:

                1. Suggest recipes I can make
                2. Include complete cooking instructions
                3. Suggest healthy alternatives if possible
                4. Give high-protein and quick meal options too

                Format every recipe like this:

                Recipe Name:

                Ingredients:
                - ...

                Instructions:
                1. ...
                2. ...
                3. ...

                Macros:
                1.Protein:
                2.Carbs:
                3.Fats:

                """
            )
        ]
    },
    config=config
)

print(response["messages"][-1].content)

## Recipe 1: Cottage Cheese Bagels

### Ingredients:
- 1 cup cottage cheese
- 1 bagel
- 1 egg
- Salt and seasoning to taste

### Instructions:
1. Pulse the cottage cheese in a blender until smooth.
2. Mix the cottage cheese with some flour and baking powder to create a dough.
3. Shape the dough into a bagel and bake at 350F for 15-20 minutes.
4. Top the bagel with a boiled egg, salt, and seasoning.

### Macros:
1. Protein: 30g
2. Carbs: 20g
3. Fats: 10g

## Recipe 2: Cottage Cheese Toast

### Ingredients:
- 1 cup cottage cheese
- 2 slices of bread
- 1 egg
- Salt and seasoning to taste

### Instructions:
1. Toast the bread until lightly browned.
2. Top the toast with cottage cheese, a boiled egg, and seasoning.

### Macros:
1. Protein: 25g
2. Carbs: 25g
3. Fats: 10g

## Recipe 3: Tuna Salad

### Ingredients:
- 1 can of tuna
- 1/2 cup cottage cheese
- 1/4 cup chopped apple
- 1 tablespoon lemon juice

### Instructions:
1. Mix the tuna, cottage cheese, chopped apple, and lemon juice togeth